<a href="https://colab.research.google.com/github/Pushpaajay/FMML_lab_modules/blob/main/FMML_Module_5_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Module 5 Lab 1

# Non Linear Support Vector Machines

```
Module Coordinator : Akshit Garg
```



In [ ]:
#Importing the necessary packages
from sklearn.datasets import load_iris
from sklearn import tree
from sklearn.model_selection import train_test_split

from sklearn import svm

import pandas
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import matplotlib.pyplot as plt


The topic of classifier in today's lab, SVMs make for really good linear separators. Let us look at an example which has linearly separable data:

In [ ]:
np.random.seed(0)

ar = np.vstack(     [\
                    np.random.multivariate_normal(np.array([1, 1]), 1.5 * np.array([[2, -1], [-1, 2.0]]), size = 50, ),\
                    np.random.multivariate_normal(np.array([3, 3]), 2 * np.array([[0.75, -0.5], [-0.5, 0.75]]), size = 50, )
                    ]\
              )

testAr = np.vstack(   [\
                      np.random.multivariate_normal(np.array([1, 1]), np.array([[0.5, -0.25], [-0.25, 0.5]]), size = 500, ),\
                      np.random.multivariate_normal(np.array([3, 3]), np.array([[0.75, -0.5], [-0.5, 0.75]]), size = 500, )
                      ]\
                  )
testy = np.array([0] * int((testAr.shape[0]/2)) + [1] * int((testAr.shape[0]/2)))

X = ar
y = np.array([0] * int((ar.shape[0]/2)) + [1] * int((ar.shape[0]/2)))

def plotDecisionBoundary(X, y, pair, clf):
  x_min, x_max = X[:, pair[0]].min() - 1, X[:, pair[0]].max() + 1
  y_min, y_max = X[:, pair[1]].min() - 1, X[:, pair[1]].max() + 1
  xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                      np.arange(y_min, y_max, 0.1))

  y_pred = clf.predict(np.c_[xx.ravel(), yy.ravel()])
  y_pred = y_pred.reshape(xx.shape)
  plt.figure(figsize=(5,4))
  plt.contourf(xx, yy, y_pred, alpha=0.4)
  plt.scatter(X[:, pair[0]], X[:, pair[1]], c = y, s = 50, edgecolor='k')
  # plt.legend()
  plt.gcf().set_dpi(130)
  plt.show()

def boundaryExp() :
  clf = svm.LinearSVC()
  pair = [0, 1]
  clf.fit(X[:, pair], y)
  plotDecisionBoundary(X, y, pair, clf)

  plt.show()

boundaryExp()

# The Kernel Trick

The true potential of SVMs is unleashed when they are combined with kernels.

## Kernels : An intuitive explanation

Kernel methods are essentially counting on using the training data (say $i^{th}$ example $(x_i, y_i)$ ) itself in a more straightforward way and learning a corresponding weight ($w_i$) for that example. Rather than trying to learn a fixed set of parameters which is done typically.
Depending on the kind of kernel used, we can virtually project the training data in a higher dimension to make it easier for the classifier to classify them.


In [ ]:
from sklearn.datasets import make_circles
X, y = make_circles(200, factor=.1, noise=.1)

clf = svm.SVC(kernel='linear').fit(X, y)

plt.style.use("seaborn")
plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='tab20')
plt.gcf().set_dpi(110)

In [ ]:
plt.style.use("default")
clf = svm.LinearSVC()
pair = [0, 1]
clf.fit(X[:, pair], y)
plotDecisionBoundary(X, y, [0, 1], clf)

However, if we artificially add another dimention to the dataset of the form:

$z = x^2 + y^2$
we can clearly see a hyperplane that can distinguish both the datasets:

In [ ]:
Z = np.array([[i[0]**2 + i[1]**2] for i in X])

In [ ]:
X_new = np.hstack((X, Z))

In [ ]:
import plotly.graph_objects as go


In [ ]:
fig = go.Figure(data = [go.Scatter3d(
    x = X_new[:, 0],
    y = X_new[:, 1],
    z = X_new[:, 2],
    mode = "markers",
    marker = {
        "color" : y,
        "line": {"width" : 4, "color":'DarkSlateGrey'},
        "colorscale": "viridis"},
)])

fig.show()

That simple trick has helped us to get another dimension in which the data is linearly separable by a hyperplane (in this case, a 2d plane)

---

Now let us use the rbf kernel and use an SVM Classifier.

In [ ]:
plt.style.use("default")
clf = svm.SVC()
pair = [0, 1]
clf.fit(X[:, pair], y)
plotDecisionBoundary(X, y, [0, 1], clf)

Now let us get back to our original dataset of iris and see if this kernel trick has helped us:

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 0)
clf = svm.SVC()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy of the decision tree on the test set: \n\n{:.3f}\n\n".format(accuracy_score(y_pred, y_test)))

Certainly, using a kernel has increased our accuracy on the iris dataset.

Things to try out:
- Try to run the same experiment after filtering different features, 2 at a time.
- Using different kinds of kernels for the SVM among: {?linear?, ?poly?, ?rbf?, ?sigmoid?, ?precomputed?}

question1

When you're experimenting with SVMs and different kernels, especially when filtering different features, it's important to carefully design your experiments. The process typically involves:

Feature Selection/Filtering: Choose two features at a time and create a subset of your dataset with only those features.

SVM Training and Testing: Train SVM models using different kernels on the filtered dataset and evaluate their performance using cross-validation or a separate test set.

Performance Evaluation: Compare the performance of different SVM models with various kernels based on metrics like accuracy, precision, recall, F1-score, or other relevant measures.

Here's a simple example using Python and scikit-learn:

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn import svm, datasets
from sklearn.feature_selection import SelectKBest, f_classif

# Load dataset (replace this with your dataset)
iris = datasets.load_iris()
X, y = iris.data, iris.target

# Feature selection: Select 2 features at a time using ANOVA F-statistic
X_new = SelectKBest(f_classif, k=2).fit_transform(X, y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

# SVM with different kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
for kernel in kernels:
    clf = svm.SVC(kernel=kernel)
    scores = cross_val_score(clf, X_train, y_train, cv=5)
    print(f'Kernel: {kernel}, Cross-Validation Accuracy: {scores.mean()}')

# Evaluate on the test set (you may want to tune hyperparameters first)
clf = svm.SVC(kernel='rbf')
clf.fit(X_train, y_train)
test_accuracy = clf.score(X_test, y_test)
print(f'Test Set Accuracy: {test_accuracy}')


In this example, we use the ANOVA F-statistic to select two features at a time. You can replace the dataset loading and feature selection parts with your specific data and feature selection method. The key is to iterate through different kernels and evaluate their performance on the filtered dataset.

Remember to adjust parameters (like C, degree, gamma, etc.) based on the characteristics of your data. Additionally, you may want to perform a more extensive hyperparameter tuning using techniques like grid search or random search.

***QUESTION2***

Support Vector Machines (SVMs) are a type of supervised learning algorithm used for classification and regression tasks. The choice of kernel in SVM is a crucial decision, as it determines the type of decision boundary the algorithm will use to separate different classes in the input space. Here's a brief overview of the different kernel options:

Linear Kernel ('linear'):

Uses a linear decision boundary.
Suitable for linearly separable data.
Computationally less expensive compared to other kernels.
Polynomial Kernel ('poly'):

Introduces polynomial terms to the decision boundary equation.
Controlled by a parameter degree which defines the degree of the polynomial.
Useful when the decision boundary is expected to be a polynomial curve.
Radial Basis Function Kernel ('rbf' or 'gaussian'):

Uses a non-linear decision boundary.
Suitable for data that is not linearly separable.
Controlled by a parameter gamma that defines the width of the RBF kernel.
Sigmoid Kernel ('sigmoid'):

Uses a sigmoid function as the decision boundary.
Suitable for data that is not linearly separable.
Controlled by parameters gamma and coef0.
Precomputed Kernel ('precomputed'):

Assumes that you have already computed the similarity or distance matrix between all pairs of data points.
The input to the SVM model should be a precomputed kernel matrix.
The choice of the kernel depends on the nature of the data and the problem at hand. Here are some general guidelines:

Use the Linear Kernel when the data is linearly separable or when there are many features compared to the number of samples.

Use the Polynomial Kernel when the decision boundary is expected to be a polynomial curve. The degree of the polynomial can be adjusted based on the complexity of the data.

Use the RBF Kernel when dealing with non-linearly separable data. The gamma parameter controls the shape of the decision boundary.

Use the Sigmoid Kernel when the data distribution is more complex and the decision boundary has a sigmoid shape.

The Precomputed Kernel is useful when you have a precomputed similarity or distance matrix between samples.

It's often a good practice to experiment with different kernels and their hyperparameters to see which one performs best for a specific dataset. Cross-validation can be used to assess the performance of different kernel choices.






